# Mini-Projeto: DataView - Exploração e Análise de Dados de Vendas
**Desenvolvedor(a) em IA para Análise Preditiva [T2] - Módulo 1 - Semana 08**

Este notebook apresenta a solução completa para o mini-projeto avaliativo de ETL e Análise Exploratória de Dados (EDA) de Vendas. O fluxo lê, limpa, transforma, analisa e visualiza dados de vendas, gerando relatórios consolidados em CSV e JSON, além de gráficos automatizados.

--- 
## Passo 1: Bloco de Importação de Bibliotecas
Importação de todas as bibliotecas padrão e de terceiros que serão utilizadas nas análises ecológicas.

In [1]:
import pandas as pd
import numpy as np
import random
import os
import json
import re
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

print("Bibliotecas importadas com sucesso!")

Bibliotecas importadas com sucesso!


--- 
## RF01 – Criar ou Carregar o Dataset de Vendas
Geração de dados sintéticos intencionalmente "sujos" para praticar a limpeza no RF03.

In [2]:
def gerar_dataset_vendas(n_registros=150, seed=42):
    """Gera um dataset sintético de vendas com dados propositalmente sujos,
    incluindo valores nulos, strings sujas, datas inválidas e outliers."""
    random.seed(seed)
    np.random.seed(seed)
    
    produtos = ['Notebook', 'Smartphone', 'Tablet', 'Monitor', 'Teclado', 'Mouse']
    precos = { 'Notebook': 3500, 'Smartphone': 2200, 'Tablet': 1800, 'Monitor': 1200, 'Teclado': 250, 'Mouse': 120 }
    categorias = { 
        "Notebook": "Computadores", "Smartphone": "Celulares",
        "Tablet": "Celulares", "Monitor": "Computadores", "Teclado": "Periféricos",
        "Mouse": "Periféricos" 
    }
    regioes = ["Sudeste", "Sul", "Nordeste", "Centro-Oeste", "Norte"]
    clientes = [f"Cliente_{i:03d}" for i in range(1, 31)]
    data_inicio = datetime(2024, 1, 1)
    dados = []
    
    for i in range(n_registros):
        produto = random.choice(produtos)
        quantidade = random.randint(1, 10)
        preco = precos[produto]
        data = data_inicio + timedelta(days=random.randint(0, 364))
        
        # Inserindo dados intencionalmente sujos para limpeza
        if random.random() < 0.05:
            quantidade = None # valor nulo
        if random.random() < 0.04:
            preco = None # valor nulo
        if random.random() < 0.03:
            produto = "   " + produto + " " # espaço extra (string suja)
            
        data_str = data.strftime("%Y-%m-%d") if random.random() > 0.02 else "DATA INVALIDA"
        
        # Geradores de outliers
        if random.random() < 0.04 and quantidade is not None:
            quantidade = quantidade * 10
        if random.random() < 0.03 and preco is not None:
            preco = preco * 5

        dados.append({
            "id_venda": i + 1,
            "data_venda": data_str,
            "cliente": random.choice(clientes),
            "produto": produto,
            "categoria": categorias.get(produto.strip() if pd.notna(produto) else "", "Outros"),
            "regiao": random.choice(regioes),
            "quantidade": quantidade,
            "preco_unitario": preco
        })
    return pd.DataFrame(dados)

df_bruto = gerar_dataset_vendas()
os.makedirs('data/raw', exist_ok=True)
df_bruto.to_csv("data/raw/vendas.csv", index=False)
print(f"Dataset gerado com {len(df_bruto)} registros e salvo em data/raw/vendas.csv.")
df_bruto.head()

Dataset gerado com 150 registros e salvo em data/raw/vendas.csv.


,id_venda,data_venda,cliente,produto,categoria,regiao,quantidade,preco_unitario
0,1,2024-01-13,Cliente_019,Mouse,Periféricos,Centro-Oeste,2.0,120.0
1,2,2024-02-17,Cliente_008,Notebook,Computadores,Centro-Oeste,1.0,3500.0
2,3,2024-01-04,Cliente_004,Teclado,Periféricos,Sudeste,5.0,250.0
3,4,2024-07-02,Cliente_013,Monitor,Computadores,Sudeste,2.0,1200.0
4,5,2024-11-17,Cliente_010,Teclado,Periféricos,Sudeste,5.0,250.0


--- 
## RF02 – Inspecionar e Descrever os Dados
Exibir formato, colunas, dtypes, valores nulos e estatísticas descritivas básicas.

In [8]:
def inspecionar_dados(df):
    """Exibe informações básicas do DataFrame."""
    print("\n=== INSPEÇÃO INICIAL DO DATASET ===")
    print(f"Shape: {df.shape}")
    print(f"\nColunas: {list(df.columns)}")
    print(f"\nTipos de dados:\n{df.dtypes}")
    print(f"\nValores nulos por coluna:\n{df.isnull().sum()}")
    print(f"\nPrimeiros registros:\n{df.head()}")
    return df.describe(include="all")

inspecionar_dados(df_bruto)


=== INSPEÇÃO INICIAL DO DATASET ===
Shape: (150, 8)

Colunas: ['id_venda', 'data_venda', 'cliente', 'produto', 'categoria', 'regiao', 'quantidade', 'preco_unitario']

Tipos de dados:
id_venda            int64
data_venda         object
cliente            object
produto            object
categoria          object
regiao             object
quantidade        float64
preco_unitario    float64
dtype: object

Valores nulos por coluna:
id_venda          0
data_venda        0
cliente           0
produto           0
categoria         0
regiao            0
quantidade        6
preco_unitario    2
dtype: int64

Primeiros registros:
   id_venda  data_venda      cliente       produto     categoria  \
0         1  2024-01-13  Cliente_019         Mouse   Periféricos   
1         2  2024-02-17  Cliente_008     Notebook   Computadores   
2         3  2024-01-04  Cliente_004       Teclado   Periféricos   
3         4  2024-07-02  Cliente_013       Monitor  Computadores   
4         5  2024-11-17  Cliente

,id_venda,data_venda,cliente,produto,categoria,regiao,quantidade,preco_unitario
count,150.000000,150,150,150,150,150,144.000000,148.000000
unique,NaN,124,30,11,3,5,NaN,NaN
top,NaN,2024-12-13,Cliente_008,Teclado,Computadores,Norte,NaN,NaN
freq,NaN,3,11,33,58,36,NaN,NaN
mean,75.500000,NaN,NaN,NaN,NaN,NaN,6.277778,1800.337838
std,43.445368,NaN,NaN,NaN,NaN,NaN,7.154093,2227.824177
min,1.000000,NaN,NaN,NaN,NaN,NaN,1.000000,120.000000
25%,38.250000,NaN,NaN,NaN,NaN,NaN,3.000000,250.000000
50%,75.500000,NaN,NaN,NaN,NaN,NaN,5.000000,1225.000000
75%,112.750000,NaN,NaN,NaN,NaN,NaN,8.000000,2200.000000


--- 
## RF03 – Limpar e Tratar os Dados
Tratar strings, converter datas inválidas para `NaT` e removê-las, filtrar nulos em colunas obrigatórias e forçar tipos numéricos corretos.

In [9]:
def limpar_strings_regex(df, colunas):
    """Usa expressões regulares para normalizar colunas de texto: colapsa múltiplos espaços internos e apara as pontas."""
    df = df.copy()
    for col in colunas:
        df[col] = df[col].apply(
            lambda s: re.sub(r"\s+", " ", str(s)).strip() if pd.notna(s) else s
        )
    return df

def limpar_dados(df):
    """Garante a validação das datas, limpeza de texto com regex, remoção de nulos cruciais e tipos coerentes."""
    df = df.copy()
    n_inicial = len(df)
    relatorio = {}
    
    # Etapa 1: Limpeza de strings
    colunas_texto = df.select_dtypes(include="object").columns
    df = limpar_strings_regex(df, colunas_texto)
    
    # Etapa 2: Conversão de datas
    df["data_venda"] = pd.to_datetime(df["data_venda"], errors="coerce")
    relatorio["datas_invalidas_removidas"] = int(df["data_venda"].isnull().sum())
    df = df.dropna(subset=["data_venda"])
    
    # Etapa 3: Remoção de nulos cruciais
    n_antes = len(df)
    df = df.dropna(subset=["quantidade", "preco_unitario"])
    relatorio["linhas_nulas_removidas"] = n_antes - len(df)
    
    # Etapa 4: Tipos numéricos corretos
    df["quantidade"] = df["quantidade"].astype(int)
    df["preco_unitario"] = df["preco_unitario"].astype(float)
    
    # Relatório Consolidador
    relatorio["registros_iniciais"] = n_inicial
    relatorio["registros_finais"] = len(df)
    relatorio["registros_removidos_total"] = n_inicial - len(df)
    
    print("=== RELATORIO DE LIMPEZA ===")
    for k, v in relatorio.items():
        print(f"{k}: {v}")

    return df, relatorio 

In [10]:
df_v1, relatorio = limpar_dados(df_bruto)
os.makedirs("data/processed/v1_com_outliers", exist_ok=True)
df_v1.to_csv("data/processed/v1_com_outliers/vendas_v1.csv", index=False)
print("\nVersão v1 com outliers salva em data/processed/v1_com_outliers/vendas_v1.csv")

=== RELATORIO DE LIMPEZA ===
datas_invalidas_removidas: 0
linhas_nulas_removidas: 8
registros_iniciais: 150
registros_finais: 142
registros_removidos_total: 8

Versão v1 com outliers salva em data/processed/v1_com_outliers/vendas_v1.csv


--- 
## RF04 – Detectar e Tratar Outliers (versões v1 e v2)
Usando o método IQR (amplitude interquartil) nas colunas de quantidade e receita.

In [11]:
def tratar_outliers(df, colunas, fator=1.5, metodo='remover'):
    """Remove ou limita outliers usando IQR."""
    df = df.copy()
    for col in colunas:
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        lim_inf = q1 - fator * iqr
        lim_sup = q3 + fator * iqr
        
        n_out = ((df[col] < lim_inf) | (df[col] > lim_sup)).sum()
        print(f'{col}: {n_out} outliers detectados (lim_inf={lim_inf:.2f}, lim_sup={lim_sup:.2f})')
        
        if metodo == 'remover':
            df = df[(df[col] >= lim_inf) & (df[col] <= lim_sup)]
        else:
            df[col] = df[col].clip(lower=lim_inf, upper=lim_sup)
    return df

In [12]:
# v2 será gerada aplicando o filtro de outliers sobre x_receita_total temporário
df_v1_tmp = df_v1.copy()
df_v1_tmp["receita_total"] = df_v1_tmp["quantidade"] * df_v1_tmp["preco_unitario"]

df_v2 = tratar_outliers(
    df_v1_tmp, 
    colunas=["quantidade", "receita_total"],
    metodo='remover'
)
df_v2 = df_v2.drop(columns=["receita_total"])

print(f"\nv1 = {len(df_v1)} linhas (com outliers)")
print(f"v2 = {len(df_v2)} linhas (outliers removidos)")
print(f"Diferença = {len(df_v1) - len(df_v2)} linhas removidas pelo IQR")

os.makedirs("data/processed/v2_outliers_tratado", exist_ok=True)
df_v2.to_csv("data/processed/v2_outliers_tratado/vendas_v2.csv", index=False)

quantidade: 2 outliers detectados (lim_inf=-4.50, lim_sup=15.50)
receita_total: 4 outliers detectados (lim_inf=-18056.25, lim_sup=33393.75)

v1 = 142 linhas (com outliers)
v2 = 136 linhas (outliers removidos)
Diferença = 6 linhas removidas pelo IQR


--- 
## RF05 – Criar Colunas Derivadas com Transformações
Enriquecer o dataset calculando receita por linha, extraindo mês, trimestre e ano, bem como categorizando o valor da venda utilizando o método vetorizado `np.select`.

In [13]:
def criar_colunas_derivadas(df):
    df = df.copy()
    df["receita_total"] = df["quantidade"] * df["preco_unitario"]
    df["mes"] = df["data_venda"].dt.month
    df["trimestre"] = df["data_venda"].dt.quarter.apply(lambda q: f"Q{q}")
    df["ano"] = df["data_venda"].dt.year
    
    # Condições usando np.select
    condicoes = [
        df["receita_total"] < 500,
        (df["receita_total"] >= 500) & (df["receita_total"] < 5000),
        df["receita_total"] >= 5000,
    ]
    rotulos = ["Baixo Valor", "Médio Valor", "Alto Valor"]
    df["faixa_receita_item"] = np.select(condicoes, rotulos, default="N/D")
    return df

# Aplicar sobre os dados limpos sem outliers (v2)
df_final = criar_colunas_derivadas(df_v2)
df_final.head()

,id_venda,data_venda,cliente,produto,categoria,regiao,quantidade,preco_unitario,receita_total,mes,trimestre,ano,faixa_receita_item
0,1,2024-01-13,Cliente_019,Mouse,Periféricos,Centro-Oeste,2,120.0,240.0,1,Q1,2024,Baixo Valor
1,2,2024-02-17,Cliente_008,Notebook,Computadores,Centro-Oeste,1,3500.0,3500.0,2,Q1,2024,Médio Valor
2,3,2024-01-04,Cliente_004,Teclado,Periféricos,Sudeste,5,250.0,1250.0,1,Q1,2024,Médio Valor
3,4,2024-07-02,Cliente_013,Monitor,Computadores,Sudeste,2,1200.0,2400.0,7,Q3,2024,Médio Valor
4,5,2024-11-17,Cliente_010,Teclado,Periféricos,Sudeste,5,250.0,1250.0,11,Q4,2024,Médio Valor


--- 
## RF06 – Calcular Métricas Agregadas (groupby)
Agregações por mês, produto (top 5), categoria e região.

In [14]:
def calcular_metricas(df):
    metricas = {}
    # Por mês
    metricas["por_mes"] = df.groupby("mes").agg(
        receita_total=("receita_total", "sum"),
        quantidade=("quantidade", "sum"),
        n_vendas=("id_venda", "count")
    ).reset_index().sort_values("mes")
    
    # Top 5 produtos
    metricas["top_produtos"] = df.groupby("produto")["receita_total"].sum().sort_values(ascending=False).head(5).reset_index()
    
    # Por categoria
    metricas["por_categoria"] = df.groupby("categoria")["receita_total"].sum().reset_index().sort_values("receita_total", ascending=False)
    
    # Por região
    metricas["por_regiao"] = df.groupby("regiao").agg(
        receita_total=("receita_total", "sum"),
        media_ticket=("receita_total", "mean")
    ).reset_index().sort_values("receita_total", ascending=False)
    
    return metricas

metricas = calcular_metricas(df_final)
for nome, tabela in metricas.items():
    print(f"\n=== {nome.upper()} ===")
    print(tabela)


=== POR_MES ===
    mes  receita_total  quantidade  n_vendas
0     1       124050.0          65        13
1     2       116240.0          62        10
2     3        50500.0          46         9
3     4        24670.0          27         5
4     5        96650.0          65        11
5     6        66540.0          47        10
6     7       150530.0          61        12
7     8        91300.0          80        13
8     9       119150.0          86        15
9    10        90980.0          43         9
10   11       122750.0          83        15
11   12        89600.0          78        14

=== TOP_PRODUTOS ===
      produto  receita_total
0    Notebook       518000.0
1  Smartphone       270600.0
2     Monitor       156000.0
3      Tablet       129600.0
4     Teclado        57000.0

=== POR_CATEGORIA ===
      categoria  receita_total
1  Computadores       674000.0
0     Celulares       400200.0
2   Periféricos        68760.0

=== POR_REGIAO ===
         regiao  receita_total  med

--- 
## RF07 – Segmentar Clientes por Nível de Gasto
Gasto Abaixo de 5k (Bronze), de 5k a 15k (Prata) e acima de 15k (Ouro) aplicando função lambda.

In [15]:
def segmentar_clientes(df):
    clientes_df = df.groupby("cliente")["receita_total"].sum().reset_index()
    clientes_df.columns = ["cliente", "total_gasto"]
    
    # Lambda com lógica de gastos para classificação do segmento
    clientes_df["segmento"] = clientes_df["total_gasto"].apply(
        lambda g: "Ouro" if g > 15000 else ("Prata" if g >= 5000 else "Bronze")
    )
    return clientes_df.sort_values("total_gasto", ascending=False)

clientes = segmentar_clientes(df_final)
print(clientes.head(10))
print(f"\nDistribuição de Segmentos:\n{clientes['segmento'].value_counts()}")

        cliente  total_gasto segmento
7   Cliente_008      99540.0     Ouro
12  Cliente_013      87200.0     Ouro
25  Cliente_026      70400.0     Ouro
8   Cliente_009      61950.0     Ouro
20  Cliente_021      61750.0     Ouro
1   Cliente_002      56650.0     Ouro
2   Cliente_003      47400.0     Ouro
21  Cliente_022      47100.0     Ouro
9   Cliente_010      46310.0     Ouro
3   Cliente_004      45220.0     Ouro

Distribuição de Segmentos:
Ouro     25
Prata     5
Name: segmento, dtype: int64


--- 
## RF08 – Calcular Estatísticas com NumPy
Cálculo de estatísticas vetorizadas sobre as receitas totais usando o NumPy.

In [16]:
def calcular_estatisticas_numpy(df):
    receitas = df["receita_total"].to_numpy()
    stats = {
        "media": float(np.mean(receitas)),
        "mediana": float(np.median(receitas)),
        "desvio_padrao": float(np.std(receitas)),
        "total": float(np.sum(receitas)),
        "p25": float(np.percentile(receitas, 25)),
        "p75": float(np.percentile(receitas, 75)),
    }
    
    # Broadcasting e percentual
    receitas_pct = (receitas / receitas.sum()) * 100
    
    # Boolean indexing
    acima_da_media = int((receitas > stats["media"]).sum())
    stats["acima_da_media"] = acima_da_media
    
    return stats, receitas_pct

stats, receitas_pct = calcular_estatisticas_numpy(df_final)
print(stats)

{'media': 8404.117647058823, 'mediana': 6600.0, 'desvio_padrao': 8084.681909289596, 'total': 1142960.0, 'p25': 1200.0, 'p75': 14000.0, 'acima_da_media': 57}


--- 
## RF09 – Criar Visualizações com Matplotlib e Seaborn
Desenvolver e salvar gráficos: tendência de vendas, ranking de produtos e dispersão regional de vendas.

In [17]:
def gerar_visualizacoes(df, metricas, output_dir="outputs/graficos"):
    os.makedirs(output_dir, exist_ok=True)
    sns.set_theme(style="whitegrid", palette="muted")
    
    # 1. Linha
    fig, ax = plt.subplots(figsize=(10, 5))
    pm = metricas["por_mes"]
    ax.plot(pm["mes"], pm["receita_total"], marker="o", color="#3b82f6", linewidth=2)
    ax.set_title("Tendência de Receita por Mês")
    ax.set_xlabel("Mês")
    ax.set_ylabel("Receita (R$)")
    fig.tight_layout()
    fig.savefig(f"{output_dir}/receita_por_mes.png", dpi=120)
    plt.close()
    
    # 2. Barras Horizontais
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.barplot(data=metricas["top_produtos"], y="produto", x="receita_total", ax=ax, hue="produto", legend=False)
    ax.set_title("Top 5 Produtos em Receita")
    ax.set_xlabel("Receita Total (R$)")
    fig.tight_layout()
    fig.savefig(f"{output_dir}/top_produtos.png", dpi=120)
    plt.close()
    
    # 3. Boxplot por Região
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.boxplot(data=df, x="regiao", y="receita_total", ax=ax, hue="regiao", legend=False)
    ax.set_title("Distribuição de Receita por Região")
    fig.tight_layout()
    fig.savefig(f"{output_dir}/dist_regiao.png", dpi=120)
    plt.close()
    
    print("Gráficos exportados com sucesso!")

gerar_visualizacoes(df_final, metricas)

Gráficos exportados com sucesso!


--- 
## RF10 – Funções de Ordem Superior
Demonstração do conceito de callback aplicando transformações personalizadas em colunas via lambdas.

In [18]:
def aplicar_transformacao(df, coluna, funcao):
    """Função de ordem superior que aceita um callback para transformação de coluna."""
    df = df.copy()
    df[f"{coluna}_transformado"] = df[coluna].apply(funcao)
    return df

# Exemplo 1: Classificar
df_demo = aplicar_transformacao(df_final, "receita_total", lambda x: "Alto" if x > 5000 else "Normal")
print(df_demo[["receita_total", "receita_total_transformado"]].head())

# Exemplo 2: Milhares
df_demo2 = aplicar_transformacao(df_final, "receita_total", lambda x: round(x / 1000, 2))
print(df_demo2[["receita_total", "receita_total_transformado"]].head())

   receita_total receita_total_transformado
0          240.0                     Normal
1         3500.0                     Normal
2         1250.0                     Normal
3         2400.0                     Normal
4         1250.0                     Normal
   receita_total  receita_total_transformado
0          240.0                        0.24
1         3500.0                        3.50
2         1250.0                        1.25
3         2400.0                        2.40
4         1250.0                        1.25


--- 
## RF11 – Ler e Escrever Arquivos (CSV e JSON)
Salvando os principais resultados em formatos legíveis com validação.

In [19]:
def exportar_resultados(metricas, clientes, stats):
    os.makedirs("outputs", exist_ok=True)
    metricas["por_mes"].to_csv("outputs/metricas_por_mes.csv", index=False, encoding="utf-8-sig")
    clientes.to_csv("outputs/segmentacao_clientes.csv", index=False, encoding="utf-8-sig")
    
    # Converter estatísticas NumPy
    stats_serializaveis = {k: round(float(v), 2) for k, v in stats.items()}
    with open("outputs/estatisticas_gerais.json", "w", encoding="utf-8") as f:
        json.dump(stats_serializaveis, f, indent=2, ensure_ascii=False)
        
    # Confirmando gravação do JSON
    with open("outputs/estatisticas_gerais.json", "r", encoding="utf-8") as f:
        lido = json.load(f)
    print("JSON lido com sucesso para validação:")
    print(json.dumps(lido, indent=2))

exportar_resultados(metricas, clientes, stats)

JSON lido com sucesso para validação:
{
  "media": 8404.12,
  "mediana": 6600.0,
  "desvio_padrao": 8084.68,
  "total": 1142960.0,
  "p25": 1200.0,
  "p75": 14000.0,
  "acima_da_media": 57.0
}


--- 
## RF12 – Consolidar a Análise e Salvar o Dataset Final
Salvar os dados finais limpos e sem outliers para explorações preditivas futuras sobre a pasta `data/final/`.

In [20]:
os.makedirs("data/final", exist_ok=True)
df_final.to_csv("data/final/vendas_final.csv", index=False)
print("Processamento concluído! O dataset final foi gravado com sucesso em data/final/vendas_final.csv.")

Processamento concluído! O dataset final foi gravado com sucesso em data/final/vendas_final.csv.
